### 문항 1 네이버 연관검색어 수집 함수 만들기

-함수명: get_related_keywords(keyword)

-반환: 문자열 리스트

-Selenium 사용 금지. Network 탭에서 요청을 찾아 requests로 재현할 것

-결과가 없으면 빈 리스트를 돌려줄 것 (예외를 던지지 말 것)

-힌트.md에 문서 찾기 힌트, 결과 파싱 힌트 있음

In [3]:
import requests
import json
import pandas as pd

In [59]:

def get_related_keywords(keyword):
    res = requests.get('https://ac.search.naver.com/nx/ac', params={'st':'100', 'q': keyword})
    res.raise_for_status

    data = res.json()
    data['items'][0]
    related_keywords = []
    for item in data['items'][0]:
        related_keywords.append(item[0])
    return related_keywords

get_related_keywords('부트캠프')

['부트캠프',
 '부트캠프 뜻',
 '부트캠프 취업',
 'ai 부트캠프',
 '직무부트캠프',
 '코햄 부트캠프',
 '마케팅 부트캠프',
 '맥북 부트캠프',
 '코멘토 부트캠프',
 '넷플릭스 부트캠프']

### 문항 2 네이버 웹툰 전체 목록 수집

In [ ]:
# 대상: https://comic.naver.com/webtoon

# 추출 필드: 제목 / 링크 / 요일

# 모든 요일의 웹툰을 수집할 것

# 링크는 상세 페이지로 바로 이동할 수 있는 절대 주소로 만들 것 (힌트 1)

# 결과를 naver_webtoon.csv로 저장할 것

In [ ]:
URL = 'https://comic.naver.com/api/webtoon/titlelist/weekday'
DETAIL_URL = 'https://comic.naver.com/webtoon/list?titleId='

WEEKDAY_KR = {
    'MONDAY': '월', 'TUESDAY': '화', 'WEDNESDAY': '수', 'THURSDAY': '목',
    'FRIDAY': '금', 'SATURDAY': '토', 'SUNDAY': '일',
}

res = requests.get(URL, params={'order': 'user'})
res.raise_for_status
data = res.json()
weekly_list = []
datas = data["titleListMap"].items()
for key, webtoons in datas:
    day = WEEKDAY_KR['key']
    for webtoon in webtoons:
        weekly_list.append({
            '제목': webtoon.get('titleName', ''),
            '링크': DETAIL_URL + str(webtoon.get('titleId', '')),
            '요일': day,
        })
weekly_list

    

In [ ]:
datas

In [63]:
# 정리해서 최적화를 하자.
URL = 'https://comic.naver.com/api/webtoon/titlelist/weekday'
DETAIL_URL = 'https://comic.naver.com/webtoon/list?titleId='

WEEKDAY_KR = {
    "MONDAY": "월", "TUESDAY": "화", "WEDNESDAY": "수", "THURSDAY": "목",
    "FRIDAY": "금", "SATURDAY": "토", "SUNDAY": "일",
}

def webtoon_all():
    res = requests.get(URL, params={'order': 'user'})
    res.raise_for_status
    data = res.json()
    weekly_list = []
    datas = data["titleListMap"].items()
    for key, webtoons in datas:
        day = WEEKDAY_KR[key]
        for webtoon in webtoons:
            weekly_list.append({
                '제목': webtoon.get('titleName', ''),
                '링크': DETAIL_URL + str(webtoon.get('titleId', '')),
                '요일': day,
            })
    return weekly_list

weekly_list = webtoon_all()
df = pd.DataFrame(weekly_list).drop_duplicates(subset=['링크', '요일'])
df.to_csv('naver_webtoon.csv', index=False, encoding='utf-8-sig')
print(f'{len(df)}건 수집 · 요일 분포\n{df['요일'].value_counts()}')

775건 수집 · 요일 분포
요일
토    121
금    116
화    113
월    112
수    107
목    106
일    100
Name: count, dtype: int64


In [ ]:
# 추가 방법

In [ ]:
res.json()['titleListMap'].keys()
# match - case 사용

dict_keys(['WEDNESDAY', 'FRIDAY', 'TUESDAY', 'THURSDAY', 'SATURDAY', 'MONDAY', 'SUNDAY'])

### 문항 3 사람인 채용공고 10페이지 수집

In [ ]:
# 대상: https://www.saramin.co.kr/zf_user/jobs/public/list

# 추출 필드: 기업명 / 그룹사 / 기업종류 / 공고명 / 직무키워드 / 학력 / 경력구분 / 근무지

# 직무키워드는 리스트로 담을 것

# 값이 없는 항목은 빈 문자열 또는 빈 리스트로 둘 것 (오류로 멈추지 말 것)

# 요청 간 0.5초 이상 지연을 둘 것

# 결과를 saramin.csv로 저장할 것

In [106]:
import time

In [67]:
URL = 'https://www.saramin.co.kr/zf_user/jobs/public/list'

res = requests.get(URL, params={'page':'1','isAjaxRequest':'y'},
                   headers={'User-Agent': 'Mozilla 5.0',
                           'x-requested-with':'XMLHttpRequest' })

res.status_code

200

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(res.json()['innerHTML'])

items = soup.select('.list_item')

def get_text(tag):
    return tag.text.strip() if tag else ''

recruits = []

for item in items:
    recruits.append({
        '기업명': get_text(item.select_one('.company_nm .str_tit')), 
                '그룹사': get_text(item.select_one('.main_corp ')),
                '기업종류':get_text(item.select_one('.info_stock')),
                '공고명':get_text(item.select_one('.job_tit > a ')),
                '직무키워드':[get_text(kwd) for kwd in item.select('.job_sector span ')],
                '학력':get_text(item.select_one('.education ')),
                '경력구분':get_text(item.select_one('.career ')), 
                '근무지':get_text(item.select_one('.work_place')),
    })
recruits

In [107]:
# 정리
URL = 'https://www.saramin.co.kr/zf_user/jobs/public/list'

def get_text(tag):
    return tag.text.strip() if tag else ''

def fetch(page):
    res = requests.get(URL, params={'page':page,'isAjaxRequest':'y'},
                      headers={'User-Agent': 'Mozilla 5.0','x-requested-with':'XMLHttpRequest' } )
    res.raise_for_status
    return res.json()

def parse(data):
    soup = BeautifulSoup(res.json()['innerHTML'])
    items = soup.select('.list_item')
    recruits = []
    for item in items:
        recruits.append({
                '기업명': get_text(item.select_one('.company_nm .str_tit')), 
                '그룹사': get_text(item.select_one('.main_corp ')),
                '기업종류':get_text(item.select_one('.info_stock')),
                '공고명':get_text(item.select_one('.job_tit > a ')),
                '직무키워드':[get_text(kwd) for kwd in item.select('.job_sector span ')],
                '학력':get_text(item.select_one('.education ')),
                '경력구분':get_text(item.select_one('.career ')), 
                '근무지':get_text(item.select_one('.work_place')),
        })
    return recruits
result = []
for page in range(1,11):
    saramin = parse(fetch(page))

    result.extend(saramin)
    print(f'{page}페이지 {len(result)}건')
    time.sleep(0.5)

df = pd.DataFrame(result)
df.to_csv('saramin.csv', index=False, encoding='utf-8-sig')
print(f'최종 {len(df)}건 공고 수집 완료')

1페이지 20건
2페이지 40건
3페이지 60건
4페이지 80건
5페이지 100건
6페이지 120건
7페이지 140건
8페이지 160건
9페이지 180건
10페이지 200건
최종 200건 공고 수집 완료
